##Codebook



In [1]:
import pandas as pd, numpy as np

In [2]:
# Read CSV-file

df = pd.read_csv('data_long_format.csv')
df['Score'] = [x.replace(',', '.') for x in df['Score']]
df['Score'] = df['Score'].astype(float)

In [3]:
# Check shape (should be 64 rows, 5 columns)

print(df.shape)

(64, 5)


In [4]:
!pip install pingouin
import pingouin as pg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 3.0 MB/s eta 0:00:00


In [5]:
# ANOVA

anova_results = pg.rm_anova(data=df, dv="Score",
                  within=["Modality", "Text_type"],
                  subject="Participant_ID",
                  detailed=True)
print(anova_results)

                 Source        SS  ddof1  ddof2        MS         F     p_unc  \
0              Modality  0.160625      1     15  0.160625  7.164778  0.017245   
1             Text_type  0.000685      1     15  0.000685  0.017171  0.897486   
2  Modality * Text_type  0.000086      1     15  0.000086  0.005217  0.943375   

   p_GG_corr       ng2  eps  
0   0.017245  0.086809  1.0  
1   0.897486  0.000405  1.0  
2   0.943375  0.000051  1.0  


#### Interpretation
The modality variance is statistically significant (p = 0.017), meaning that it is easier to recall something that you have read compared to something that you have listened to. No effect for the metaphor/literal condition (text type).


#### Mixed linear model

In [6]:
!pip install statsmodels
import statsmodels.formula.api as smf

In [7]:
# Mixed linear model

df["grp"] = 1
vc = {"participant": "0 + C(Participant_ID)",
      "text":        "0 + C(Base_text)"}

model = smf.mixedlm("Score ~ Modality * Text_type",
                    data=df,
                    groups="grp",
                    vc_formula=vc,
                    re_formula="0").fit(reml=True)

print(model.summary())

                         Mixed Linear Model Regression Results
Model:                        MixedLM            Dependent Variable:            Score  
No. Observations:             64                 Method:                        REML   
No. Groups:                   1                  Scale:                         0.0187 
Min. group size:              64                 Log-Likelihood:                19.0764
Max. group size:              64                 Converged:                     Yes    
Mean group size:              64.0                                                     
---------------------------------------------------------------------------------------
                                              Coef. Std.Err.   z    P>|z| [0.025 0.975]
---------------------------------------------------------------------------------------
Intercept                                     0.526    0.048 11.031 0.000  0.432  0.619
Modality[T.Reading]                           0.098    0.

/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


In [8]:
# See how much variance is captured by participant, text and residual

vc_p  = model.vcomp[0]                  # participant variance
vc_t  = model.vcomp[1]                  # text variance
vc_e  = model.scale                     # residual variance
tot   = vc_p + vc_t + vc_e
print("\n=== Variance components ===")
for name, v in [("Participant", vc_p), ("Base_text", vc_t), ("Residual", vc_e)]:
    print(f"  {name:11s}: {v:.5f}  ({100*v/tot:4.1f}% of variance, SD={np.sqrt(v):.3f})")


=== Variance components ===
  Participant: 0.00457  (15.3% of variance, SD=0.068)
  Base_text  : 0.00651  (21.8% of variance, SD=0.081)
  Residual   : 0.01873  (62.8% of variance, SD=0.137)


In [9]:
# Inspect means of the conditions

print("\n=== Condition means ===")
print(df.groupby(["Modality","Text_type"])["Score"].mean().round(3))


=== Condition means ===
Modality   Text_type   
Listening  Literal         0.526
           Metaphorical    0.530
Reading    Literal         0.624
           Metaphorical    0.632
Name: Score, dtype: float64


Filtered; without participant 8

In [10]:
df = pd.read_csv('data_long_format_filtered.csv')
df['Score'] = [x.replace(',', '.') for x in df['Score']]
df['Score'] = df['Score'].astype(float)
print(df.shape)

(60, 5)


In [11]:
anova_results = pg.rm_anova(data=df, dv="Score",
                  within=["Modality", "Text_type"],
                  subject='Participant_ID',
                  detailed=True)
print(anova_results)

                 Source        SS  ddof1  ddof2        MS          F  \
0              Modality  0.228166      1     14  0.228166  15.372681   
1             Text_type  0.026174      1     14  0.026174   1.218242   
2  Modality * Text_type  0.000792      1     14  0.000792   0.045862   

      p_unc  p_GG_corr       ng2  eps  
0  0.001538   0.001538  0.160952  1.0  
1  0.288317   0.288317  0.021532  1.0  
2  0.833514   0.833514  0.000665  1.0  


In [12]:
df["grp"] = 1
vc = {"participant": "0 + C(Participant_ID)",
      "text":        "0 + C(Base_text)"}

model = smf.mixedlm("Score ~ Modality * Text_type",
                    data=df,
                    groups="grp",
                    vc_formula=vc,
                    re_formula="0").fit(reml=True)

print(model.summary())

                         Mixed Linear Model Regression Results
Model:                        MixedLM            Dependent Variable:            Score  
No. Observations:             60                 Method:                        REML   
No. Groups:                   1                  Scale:                         0.0104 
Min. group size:              60                 Log-Likelihood:                28.6092
Max. group size:              60                 Converged:                     Yes    
Mean group size:              60.0                                                     
---------------------------------------------------------------------------------------
                                              Coef. Std.Err.   z    P>|z| [0.025 0.975]
---------------------------------------------------------------------------------------
Intercept                                     0.510    0.045 11.389 0.000  0.422  0.598
Modality[T.Reading]                           0.121    0.

/usr/local/lib/python3.12/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


In [13]:
# See how much variance is captured by participant, text and residual

vc_p  = model.vcomp[0]                  # participant variance
vc_t  = model.vcomp[1]                  # text variance
vc_e  = model.scale                     # residual variance
tot   = vc_p + vc_t + vc_e
print("\n=== Variance components ===")
for name, v in [("Participant", vc_p), ("Base_text", vc_t), ("Residual", vc_e)]:
    print(f"  {name:11s}: {v:.5f}  ({100*v/tot:4.1f}% of variance, SD={np.sqrt(v):.3f})")


=== Variance components ===
  Participant: 0.00578  (24.5% of variance, SD=0.076)
  Base_text  : 0.00736  (31.2% of variance, SD=0.086)
  Residual   : 0.01044  (44.3% of variance, SD=0.102)


In [14]:
# Inspect means of the conditions

print("\n=== Condition means ===")
print(df.groupby(["Modality","Text_type"])["Score"].mean().round(3))


=== Condition means ===
Modality   Text_type   
Listening  Literal         0.507
           Metaphorical    0.542
Reading    Literal         0.623
           Metaphorical    0.672
Name: Score, dtype: float64
